# Usage of CamelUp Code

In [25]:
%load_ext autoreload
%autoreload 2

from CamelUp import *

### Examination of structure of the field

In [26]:
demo_players = {"Player 1":player("Player 1"),"Player 2":player("Player 2"),
                "Player 3":player("Player 3"),"Player 4":player("Player 4")}

demoE = Field([
    [], [], [], [], [], ['Yellow', 'Green'], ['Purple', 'Blue', 'Orange'], ["DESERT", "Player 1"], [], 
    ["Black", "White"], [], [], [], [],[], [], [], [], []],copy.deepcopy(demo_players))


In [27]:
demoE.game_field
mask = {"Purple":1,"Blue":2,"Orange":3,"Yellow":4,"Green":5,"White":6,"Black":7,"DESERT":8,"OASIS":9}


In [28]:
rendered_field, inv_player_mask = render_field(demoE)
print(rendered_field)
print(inv_player_mask)

[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 4  5  0  0  0  0  0]
 [ 1  2  3  0  0  0  0]
 [ 8 10  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 7  6  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]]
{10: 'Player 1', 11: 'Player 2', 12: 'Player 3', 13: 'Player 4'}


### New generation flexible for testing
Step 1: create all permutations for the dice

In [ ]:
import itertools
camels_thrown = []
camels_not_thrown = [1,2,3,4,5,6]
positions = np.zeros((5,3))
DO_hits = np.zeros((len(demoE.players),1))
verbose = True

## use time to measure performance
if verbose:
    start_time = time.time()

len_all_camels = len(camels_thrown) + len(camels_not_thrown)
draw_n_camels = len(camels_not_thrown)
if len_all_camels > 5:
    draw_n_camels -= 1
print("draw_n_camels: ", draw_n_camels)
all_dice_permutations = list(itertools.product(range(1,4), repeat=draw_n_camels))
print("Number of paths: ", len(all_dice_permutations), "first 5 paths: ", all_dice_permutations[:5], "SHOULD number of paths", 3**draw_n_camels)
# Generate all possible permutations of camels_not_thrown (numba friendly: i.e., as lists of integer arrays)
# (This is separate from all_dice_permutations, which is all product/dice rolls)
all_camel_permutations = list(itertools.permutations(camels_not_thrown))
print("Number of permutations: ", len(all_camel_permutations), "first 5 permutations:", all_camel_permutations[:5], "should permutations")

## include white camel as well as black camel:
if 5 in camels_not_thrown:
    camels_not_thrown.remove(6)
    camels_not_thrown.append(7)
    all_camel_permutations.extend(list(itertools.permutations(camels_not_thrown)))

if verbose:
    interval_time_permutations = time.time()
    print("Time to generate permutations:", round(interval_time_permutations - start_time, 6), "seconds")

# Efficiently create all combinations of camel order (permutation) and dice rolls (product) using itertools.product
all_full_permutations = list(itertools.product(all_camel_permutations, all_dice_permutations))
print("Number of full permutations: ", len(all_full_permutations), "First 5:", all_full_permutations[:5], "should be", len(all_camel_permutations)*len(all_dice_permutations))

if verbose:
    interval_time_product = time.time()
    print("Time to generate product:", round(interval_time_product - interval_time_permutations, 6), "seconds")


draw_n_camels:  5
Number of paths:  243 first 5 paths:  [(1, 1, 1, 1, 1), (1, 1, 1, 1, 2), (1, 1, 1, 1, 3), (1, 1, 1, 2, 1), (1, 1, 1, 2, 2)] SHOULD number of paths 243
Number of permutations:  720 first 5 permutations: [(1, 2, 3, 4, 5, 6), (1, 2, 3, 4, 6, 5), (1, 2, 3, 5, 4, 6), (1, 2, 3, 5, 6, 4), (1, 2, 3, 6, 4, 5)] should permutations
Time to generate permutations: 0.001256 seconds
Number of full permutations:  349920 First 5: [((1, 2, 3, 4, 5, 6), (1, 1, 1, 1, 1)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 1, 2)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 1, 3)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 2, 1)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 2, 2))] should be 349920
Time to generate product: 0.049361 seconds


Step 2: simulate all paths!

In [93]:
if verbose:
    interval_time_product = time.time()

checkmark_idx = 0; check_frequency = 50; 
## path simulation
for camel_order, dice_rolls in all_full_permutations[:100]:
    ##create field copy
    field_copy = rendered_field.copy()
    
    for move_idx in range(len(dice_rolls)):
        move = dice_rolls[move_idx]
        camel = camel_order[move_idx]
        ## find camel in field_copy
        camel_idx = tuple(np.argwhere(field_copy == camel)[0])
        ## cut out stack of camel
        # Extract nonzero integers in same row and col >= camel_idx[1]
        row = camel_idx[0]
        col = camel_idx[1]
        # Extract the stack as a vector (all nonzero from (row, col) to (row, end))
        # Optimized: slice out directly using numpy for speed (all leading nonzero, so from col to first 0 or end)
        # Directly operate on the field_copy row to avoid extra reference
        end = (field_copy[row] != 0).sum()

        stack = field_copy[row, col:end].copy().reshape(1, -1)
        # Zero out in-place
        field_copy[row, col:end] = 0
        ## move camel
        below_target = False
        if camel not in [6,7]:
            target_field = row + move
            if field_copy[target_field,0] == 8:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field -= 1
                below_target = True
            elif field_copy[target_field,0] == 9:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field += 1
        else:
            target_field = row - move
            if field_copy[target_field,0] == 8:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field += 1
                below_target = True
            elif field_copy[target_field,0] == 9:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field -= 1

        ##place camel on or below target
        #print(stack,field_copy[target_field,:-stack.shape[1]].reshape(1,-1))
        if below_target:
            field_copy[target_field, :] = np.concatenate((stack, field_copy[target_field, :-stack.shape[1]].reshape(1, -1)), axis=1)
        else:
            end_target_field = (field_copy[target_field] != 0).sum()
            field_copy[target_field,end_target_field:end_target_field+stack.shape[1]] = stack
        
        if target_field > 16:
            break
    camel_positions = np.zeros((5,1))
    ## evaluation of the path
    for camel in range(1,6):
        camel_idx = tuple(np.argwhere(field_copy == camel)[0])
        row = camel_idx[0]
        col = camel_idx[1]
        camel_positions[camel-1,0] = row*7+col
    camel_ranking = np.argsort(-camel_positions.flatten())
    positions[camel_ranking[0],0] += 1
    positions[camel_ranking[1],1] += 1
    for i in range(2,5):
        positions[camel_ranking[i],2] += 1

calc_time = time.time()
print(f"Calculation time: {calc_time - interval_time_product:.6f} seconds")

end_time = time.time()
print(f"Operation time: {end_time - start_time:.6f} seconds")

Calculation time: 0.027429 seconds
Operation time: 444.974888 seconds


### put together in a numba accelerated function
Done

In [202]:
for i in range(12):
    positions, DO_hits = sim_all_moves(rendered_field, 4, n_camels_thrown, camels_not_thrown, verbose=False)
positions, DO_hits

(array([[ 47710,  59492, 242718],
        [ 64964, 100506, 184450],
        [113150,  83918, 152852],
        [ 38904,  47596, 263420],
        [ 85192,  58408, 206320]], dtype=int64),
 array([[494916],
        [     0],
        [     0],
        [     0]], dtype=int64))

In [203]:
for i in range(12):
    positions, DO_hits = sim_all_moves(rendered_field, 4, n_camels_thrown, [1,2,3,4,5], verbose=False)
rendered_field, positions, DO_hits 

(array([[ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  0,  0,  0,  0],
        [ 8, 10,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 7,  6,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0]]),
 array([[ 4327,  4892, 19941],
        [ 6382,  8186, 14592],
        [ 8918,  7496, 12746],
        [ 3208,  3602, 22350],
        [ 6325,  4984, 17851]], dtype=int64),
 array([[37428],
        [    0],
        [    0],
        [    0]], dtype=int64))

bla

In [223]:
len("════════════════════════                                                                    ════════════════════════")

116

In [220]:
[i[24:-24] for i in """
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║             <16>               < 1>               < 2>             ║      xyz----zyx      ║
║      xyz----zyx      ║     <15>                                                  < 3>     ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║   <14>                                                      < 4>   ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
════════════════════════                                                                    ════════════════════════
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                 ═════════════════════════════════                  ║      xyz----zyx      ║
║      xyz----zyx      ║   <13>          ║Value of Information:     {VOI:5.3f}║           < 5>   ║      xyz----zyx      ║
║      xyz----zyx      ║                 ═════════════════════════════════                  ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
════════════════════════                                                                    ════════════════════════
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║   <12>                                                      < 6>   ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║     <11>                                                  < 7>     ║      xyz----zyx      ║
║      xyz----zyx      ║             <10>               < 9>               < 8>             ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║""".split("\n")]

['',
 '                                                                    ',
 '             <16>               < 1>               < 2>             ',
 '     <15>                                                  < 3>     ',
 '                                                                    ',
 '                                                                    ',
 '   <14>                                                      < 4>   ',
 '                                                                    ',
 '                                                                    ',
 '                                                                    ',
 '                                                                    ',
 '                 ═════════════════════════════════                  ',
 '   <13>          ║Value of Information:     {VOI:5.3f}║           < 5>   ',
 '                 ═════════════════════════════════                  ',
 '                                       

VOI calculation blueprint

In [24]:
import numpy as np
game_inventory_matrix = np.array([[1,1,2],[0,0,1],[0,1,2],[0,0,2],[1,1,2]])
path_multiplier = 21

positions = np.ones((path_multiplier,5,3))
positions[:-6,:,:]*=2
##create a 3D array of the game inventory matrix, repeated for each paths first element
game_inventory_multiplier = np.tile(game_inventory_matrix, (positions.shape[0],1,1))

## calculate the number of paths for each first element (color+die result)
n_paths = positions.sum(axis = 2)[:,0]

#print(n_paths)
VOI_array = np.matmul(
    positions * (1/n_paths)[:, np.newaxis, np.newaxis], 
    np.tile([[5,3,2],[1,1,1],[-1,-1,-1]], (positions.shape[0],1)).reshape(positions.shape[0],3,3))
VOI_array[VOI_array<1] = 0
VOI_array = VOI_array * game_inventory_multiplier

#print(VOI_array)

next_voi = VOI_array.sum(axis = (1,2))
del VOI_array

VOI_array_now  = np.matmul(
    (positions).sum(axis = 0)/n_paths.sum(),
    np.array([[5,3,2],[1,1,1],[-1,-1,-1]]))
#print(VOI_array_now)
VOI_array_now[VOI_array_now<1] = 0
VOI_array_now*=game_inventory_matrix
#print(VOI_array_now)
now_voi  = VOI_array_now.sum()
print(now_voi)
print(next_voi)
VOI = ((next_voi - now_voi)*n_paths).sum()/n_paths.sum()
print(VOI)

6.333333333333333
[6.33333333 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333
 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333
 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333
 6.33333333 6.33333333 6.33333333]
0.0


### Numba accelerated function tester

In [ ]:
### Numba accelerated function tester
n_camels_thrown = 0
game_inventory_matrix = np.array([[1,1,2],[0,1,2],[0,0,1],[1,1,2],[0,0,0]])
for i in range(12):
    positions, DO_hits, VOI = sim_all_moves(rendered_field, 4, n_camels_thrown, [1, 2, 3, 4, 5, 6], game_inventory_matrix, verbose=False)
rendered_field, positions, DO_hits , VOI

NameError: name 'n_camels_thrown' is not defined

### Currently unused test cases

In [ ]:
                    

demo3_players = {"Michael":player("Micheal"),"Dwight":player("Dwight"),"Jim":player("Jim")}

demoE = Field([[], [], [], [], [], [], [], [], [], ['White'], ['Blue'], [], ['Orange'], [],
               ['Yellow', 'Green'], [], [], [], []],copy.deepcopy(demo_players))

demo1 = Field([[], [], [], [], [], ['White','Blue','Orange'], ['DESERT', 'Player 4'], ['Yellow', 'Green'], 
               ['DESERT', 'Player 1'], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo_players))

demoS2 = Field([['White','Blue'], ['Yellow', 'Green'], ['Orange'], ['DESERT', 'Player 1'], [],
               ['DESERT', 'Player 4'], [], [], [], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo_players))

demo3 = Field([['White','Blue'], ['Yellow', 'Green'], ['Orange'], [], [], [], [], [], 
               [], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo3_players))

demoS = Field([['White','Blue'], ['Yellow', 'Green'], ['Orange'], [], [], [], [], [], 
               [], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo_players))


CC1 = CamelUp(4,field = demo1)
CC = CamelUp(4,field = demoE)
CCS = CamelUp(4,field = demoS)
CC3 = CamelUp(4,field = demo3)
CCS2 = CamelUp(4,field = demoS2)


dublin_players = {"Holly":player("Holly"),"Eike":player("Eike"),"Fredrik":player("Fredrik"),
                  "Joaquin":player("Joaquin")}

dublin_field = Field([['White','Blue'], ['Yellow', 'Green'], ['Orange'], [], [], [], [], [], 
                      [], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(dublin_players))

CC_dublin = CamelUp(4,field = dublin_field)
